# 01 — What the ingestion step does (walkthrough)

Companion to `src/data/ingest_taobao.py` and `src/data/dq_report.py`.
**This notebook does not run the 24 GB ingestion.** It shows the logic on the two small tables,
then displays the real pipeline's outputs if you have already run it.

In [1]:
from pathlib import Path
import sys, json, time
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw" / "taobao"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
con = duckdb.connect()
con.execute("SET memory_limit='16GB'")
con.execute("SET threads=8")
# Taobao timestamps are Beijing time: 1494000000 is exactly 2017-05-06 00:00 at UTC+8.
# Without this, DuckDB uses the machine's own timezone and the per-day buckets are shifted.
con.execute("SET TimeZone='Asia/Shanghai'")
print("project root:", ROOT)

project root: /data/home/mitanshub/Ad_ranking_Bidding_Stepwise


## 1. Why typed reading with null handling matters

Compare DuckDB's automatic type guessing with the explicit schema the ingest script uses,
on `ad_feature` (31 MB).

In [2]:
auto = con.execute(f"SELECT * FROM read_csv_auto('{RAW/'ad_feature.csv'}')").df()
print("auto-detected dtypes:"); print(auto.dtypes.to_dict())
print("nulls:", auto.isna().sum().to_dict())

auto-detected dtypes:
{'adgroup_id': dtype('int64'), 'cate_id': dtype('int64'), 'campaign_id': dtype('int64'), 'customer': dtype('int64'), 'brand': <StringDtype(na_value=nan)>, 'price': dtype('float64')}
nulls: {'adgroup_id': 0, 'cate_id': 0, 'campaign_id': 0, 'customer': 0, 'brand': 0, 'price': 0}


In [3]:
from src.data.ingest_taobao import TABLES
print(json.dumps(TABLES["ad_feature"], indent=1))
cols = ", ".join(f"'{c}': '{t}'" for c, t in TABLES["ad_feature"].items())
typed = con.execute(f"""SELECT * FROM read_csv('{RAW/'ad_feature.csv'}', header=true,
                          columns={{{cols}}}, nullstr=['NULL',''])""").df()
print("typed dtypes:"); print(typed.dtypes.to_dict())
print("nulls:", typed.isna().sum().to_dict())

{
 "adgroup_id": "BIGINT",
 "cate_id": "BIGINT",
 "campaign_id": "BIGINT",
 "customer": "BIGINT",
 "brand": "BIGINT",
 "price": "DOUBLE"
}


typed dtypes:
{'adgroup_id': dtype('int64'), 'cate_id': dtype('int64'), 'campaign_id': dtype('int64'), 'customer': dtype('int64'), 'brand': Int64Dtype(), 'price': dtype('float64')}
nulls: {'adgroup_id': 0, 'cate_id': 0, 'campaign_id': 0, 'customer': 0, 'brand': 246330, 'price': 0}


## 2. The schemas the ingest script applies
`user_profile` is renamed on the way in (`userid` → `user`) and its trailing-space header is
replaced by a clean name, so all tables can join on `user`.

In [4]:
pd.DataFrame({t: pd.Series(c) for t, c in TABLES.items()}).fillna("")

,raw_sample,behavior_log,ad_feature,user_profile
adgroup_id,BIGINT,,BIGINT,
age_level,,,,BIGINT
brand,,BIGINT,BIGINT,
btag,,VARCHAR,,
campaign_id,,,BIGINT,
cate,,BIGINT,,
cate_id,,,BIGINT,
clk,TINYINT,,,
cms_group_id,,,,BIGINT
cms_segid,,,,BIGINT


## 3. Small-table ingestion demo (written to a scratch folder, not the real output)
Same `read_csv` -> Parquet code as the script, for the two small tables only.

In [5]:
demo = ROOT / "data" / "interim" / "notebook_demo"
demo.mkdir(parents=True, exist_ok=True)
for name in ["ad_feature", "user_profile"]:
    cols = ", ".join(f"'{c}': '{t}'" for c, t in TABLES[name].items())
    dst = demo / f"{name}.parquet"
    con.execute(f"""COPY (SELECT * FROM read_csv('{RAW/(name+'.csv')}', header=true,
                          columns={{{cols}}}, nullstr=['NULL',''])) TO '{dst}' (FORMAT parquet, COMPRESSION zstd)""")
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{dst}')").fetchone()[0]
    print(f"{name:13s} rows={n:>10,d}  csv={ (RAW/(name+'.csv')).stat().st_size/1e6:7.1f} MB  parquet={dst.stat().st_size/1e6:7.1f} MB")

ad_feature    rows=   846,811  csv=   31.3 MB  parquet=    8.9 MB


user_profile  rows= 1,061,768  csv=   24.1 MB  parquet=    4.6 MB


In [6]:
con.execute(f"SELECT * FROM read_parquet('{demo/'user_profile.parquet'}') LIMIT 5").df()

,user,cms_segid,cms_group_id,final_gender_code,age_level,pvalue_level,shopping_level,occupation,new_user_class_level
0,234,0,5,2,5,<NA>,3,0,3
1,523,5,2,2,2,1,3,1,2
2,612,0,8,1,2,2,3,0,<NA>
3,1670,0,4,2,4,<NA>,1,0,<NA>
4,2545,0,10,1,4,<NA>,3,0,<NA>


## 4. What `dq_report.py` measures
| Check | Why it matters |
|---|---|
| null rate per column | decides imputation vs. dropping |
| rows per day, timestamp min/max | find rows outside the documented 8-day window |
| exact duplicate rate | duplicates inflate behaviour counts |
| events per user (p50 … max) | spot bot-like users |
| price quantiles | spot outliers |
| users with both a click and a non-click | only these can be scored by GAUC |
| impression→behaviour join coverage | precondition for every cross-surface claim |

## 5. Results of the real pipeline (shown only if you have run it)

In [7]:
pq = ROOT / "data" / "processed" / "raw_parquet"
if pq.exists():
    for f in sorted(pq.glob("*.parquet")):
        n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{f}')").fetchone()[0]
        print(f"{f.name:24s} rows={n:>13,d}  {f.stat().st_size/1e9:6.2f} GB")
else:
    print("raw_parquet/ not found — run: python -m src.data.ingest_taobao")

raw_parquet/ not found — run: python -m src.data.ingest_taobao


In [8]:
rp = ROOT / "outputs" / "results" / "dq_report.json"
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()), indent=1)[:6000])
else:
    print("dq_report.json not found — run: python -m src.data.dq_report")

dq_report.json not found — run: python -m src.data.dq_report
